In [1]:
import zipfile
from pathlib import Path
zip_path = Path("/content/DEPI-GRADUATION-PROJECT-task-2 (1).zip")
extract_path = Path("/content/project")
extract_path.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)
print("Extracted successfully!")

Extracted successfully!


In [2]:
import pandas as pd
from pathlib import Path
PROJECT_DIR = Path("/content/project/DEPI-GRADUATION-PROJECT-task-2")
DATA_DIR = PROJECT_DIR / "data/processed/price_review_task4"
train_original = pd.read_csv(DATA_DIR / "train_scored_4_1.csv")
train_clean = pd.read_csv(DATA_DIR / "train_clean_4_3.csv")
val_df = pd.read_csv(DATA_DIR / "val_scored_4_1.csv")
print("Original Train:", train_original.shape)
print("Clean Train:", train_clean.shape)
print("Validation:", val_df.shape)

Original Train: (5165, 64)
Clean Train: (5109, 64)
Validation: (1252, 64)


In [3]:
print("Original Train Columns:")
print(train_original.columns.tolist())
print("\nClean Train Columns:")
print(train_clean.columns.tolist())
print("\nValidation Columns:")
print(val_df.columns.tolist())

Original Train Columns:
['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'is_phone_listing', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known', 'storage_imp_src', 'storage_imp_conf', 'storage_imp_post', 'ram_imp_src', 'ram_imp_conf', 'ram_imp_post', 'imp_price_used', 'ref_matched', 'storage_gb_orig', 'storage_gb_was_imputed', 'ram_gb_orig', 'ram_gb_was_imputed', 'ref_n', 'ref_med_log', 'ref_mad_log', 'ref_level', 'ref_median_price', 'price_ratio', 'mad_z', 'price_review_reason', 'pri

In [4]:
print("Original Train:")
print("Rows:", len(train_original))
print("Missing prices:", train_original["price"].isna().sum())
print("\nClean Train:")
print("Rows:", len(train_clean))
print("Missing prices:", train_clean["price"].isna().sum())
print("\nValidation:")
print("Rows:", len(val_df))
print("Missing prices:", val_df["price"].isna().sum())
print("\nRemoved rows:", len(train_original) - len(train_clean))

Original Train:
Rows: 5165
Missing prices: 0

Clean Train:
Rows: 5109
Missing prices: 0

Validation:
Rows: 1252
Missing prices: 0

Removed rows: 56


In [5]:
target = "price"
# Features used for prediction
features = [
    "brand",
    "model_key",
    "storage_gb",
    "ram_gb",
    "condition",
    "has_5g",
    "has_warranty",
    "has_box",
    "family",
]

print("Features:", features)
print("\nTarget:", target)

print("\nFeature data types:")
print(train_original[features].dtypes)

print("\nMissing values in features:")
print(train_original[features].isna().sum())

Features: ['brand', 'model_key', 'storage_gb', 'ram_gb', 'condition', 'has_5g', 'has_warranty', 'has_box', 'family']

Target: price

Feature data types:
brand            object
model_key        object
storage_gb      float64
ram_gb          float64
condition        object
has_5g             bool
has_warranty       bool
has_box            bool
family           object
dtype: object

Missing values in features:
brand            276
model_key          0
storage_gb      1033
ram_gb          1077
condition       4809
has_5g             0
has_warranty       0
has_box            0
family             0
dtype: int64


In [6]:
categorical_features = [
    "brand",
    "model_key",
    "condition",
    "family"
]
X_train_original = train_original[features].copy()
X_train_clean = train_clean[features].copy()
X_val = val_df[features].copy()

y_train_original = train_original[target].copy()
y_train_clean = train_clean[target].copy()
y_val = val_df[target].copy()

for df in [X_train_original, X_train_clean, X_val]:
    for col in categorical_features:
        df[col] = df[col].fillna("Unknown").astype(str)

print("Original X:", X_train_original.shape)
print("Clean X:", X_train_clean.shape)
print("Validation X:", X_val.shape)

print("\nMissing values after preparation:")
print(X_train_original.isna().sum())

Original X: (5165, 9)
Clean X: (5109, 9)
Validation X: (1252, 9)

Missing values after preparation:
brand              0
model_key          0
storage_gb      1033
ram_gb          1077
condition          0
has_5g             0
has_warranty       0
has_box            0
family             0
dtype: int64


In [7]:
!pip install catboost
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
model_original = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Huber:delta=1.0",
    verbose=100,
    random_seed=42
)
model_original.fit(
    X_train_original,
    y_train_original,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100
)

0:	learn: 24037.5300575	test: 23521.9375660	best: 23521.9375660 (0)	total: 52.1ms	remaining: 52.1s
100:	learn: 20719.0173037	test: 20133.4052432	best: 20133.4052432 (100)	total: 435ms	remaining: 3.88s
200:	learn: 19432.7725084	test: 18826.9675057	best: 18826.9675057 (200)	total: 1.04s	remaining: 4.13s
300:	learn: 18655.8732395	test: 17955.0082802	best: 17955.0082802 (300)	total: 1.82s	remaining: 4.23s
400:	learn: 18027.2872591	test: 17263.2256564	best: 17263.2256564 (400)	total: 2.46s	remaining: 3.68s
500:	learn: 17480.2724952	test: 16666.7947803	best: 16666.7947803 (500)	total: 3.11s	remaining: 3.1s
600:	learn: 16988.0314463	test: 16110.3696453	best: 16110.3696453 (600)	total: 3.74s	remaining: 2.48s
700:	learn: 16543.7853548	test: 15590.0206737	best: 15590.0206737 (700)	total: 4.28s	remaining: 1.82s
800:	learn: 16146.9727564	test: 15119.3201042	best: 15119.3201042 (800)	total: 4.69s	remaining: 1.17s
900:	learn: 15786.3865836	test: 14676.5670808	best: 14676.5670808 (900)	total: 5.13s	r

CatBoostRegressor(depth=6, iterations=1000, learning_rate=0.05, loss_function='Huber:delta=1.0', random_seed=42, verbose=100)

In [8]:
from sklearn.metrics import mean_absolute_error
# Predictions on validation
pred_original = model_original.predict(X_val)
# Calculate MAE
mae_original = mean_absolute_error(y_val, pred_original)
print("Original Train - Validation MAE:", mae_original)

Original Train - Validation MAE: 14288.439066253393


In [9]:
model_clean = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Huber:delta=1.0",
    verbose=100,
    random_seed=42
)
model_clean.fit(
    X_train_clean,
    y_train_clean,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100
)

0:	learn: 24167.3461300	test: 23501.0788063	best: 23501.0788063 (0)	total: 4.43ms	remaining: 4.42s
100:	learn: 20914.6322604	test: 20195.0017053	best: 20195.0017053 (100)	total: 461ms	remaining: 4.1s
200:	learn: 19548.5554286	test: 18775.8495249	best: 18775.8495249 (200)	total: 908ms	remaining: 3.61s
300:	learn: 18771.5724802	test: 17928.2901519	best: 17928.2901519 (300)	total: 1.59s	remaining: 3.69s
400:	learn: 18164.0280063	test: 17294.3391944	best: 17294.3391944 (400)	total: 2.21s	remaining: 3.3s
500:	learn: 17630.7208734	test: 16742.7271113	best: 16742.7271113 (500)	total: 2.9s	remaining: 2.89s
600:	learn: 17169.1943882	test: 16255.0996250	best: 16255.0996250 (600)	total: 3.8s	remaining: 2.52s
700:	learn: 16761.8540391	test: 15820.9617941	best: 15820.9617941 (700)	total: 4.68s	remaining: 2s
800:	learn: 16388.3952454	test: 15419.4231994	best: 15419.4231994 (800)	total: 5.83s	remaining: 1.45s
900:	learn: 16040.4005012	test: 15020.0801735	best: 15020.0801735 (900)	total: 6.94s	remaini

CatBoostRegressor(depth=6, iterations=1000, learning_rate=0.05, loss_function='Huber:delta=1.0', random_seed=42, verbose=100)

In [10]:
pred_clean = model_clean.predict(X_val)
mae_clean = mean_absolute_error(y_val, pred_clean)
print("Clean Train - Validation MAE:", mae_clean)

Clean Train - Validation MAE: 14566.000517084492


In [11]:
results = pd.DataFrame({
    "Experiment": ["Original Train", "Clean Train"],
    "Train Rows": [len(train_original), len(train_clean)],
    "Validation MAE": [mae_original, mae_clean]
})
results

,Experiment,Train Rows,Validation MAE
0,Original Train,5165,14288.439066
1,Clean Train,5109,14566.000517


In [12]:
model_original_mae = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    verbose=100,
    random_seed=42
)
model_original_mae.fit(
    X_train_original,
    y_train_original,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100
)

0:	learn: 16169.9955387	test: 14897.0303427	best: 14897.0303427 (0)	total: 21ms	remaining: 21s
100:	learn: 6276.1197114	test: 6844.9471753	best: 6844.9471753 (100)	total: 1.16s	remaining: 10.4s
200:	learn: 5897.7565951	test: 6484.0133449	best: 6483.4478053 (198)	total: 2.3s	remaining: 9.13s
300:	learn: 5690.5672269	test: 6317.0364350	best: 6317.0364350 (300)	total: 4.09s	remaining: 9.49s
400:	learn: 5562.5254642	test: 6275.6447150	best: 6269.3392390 (394)	total: 5.45s	remaining: 8.14s
500:	learn: 5464.7170725	test: 6234.4563566	best: 6234.0401548 (498)	total: 6.69s	remaining: 6.66s
600:	learn: 5394.7278990	test: 6222.7714817	best: 6220.3140871 (591)	total: 8.02s	remaining: 5.33s
700:	learn: 5333.8217272	test: 6208.2824968	best: 6207.8549887 (694)	total: 9.34s	remaining: 3.98s
800:	learn: 5287.7045344	test: 6204.2837668	best: 6204.2092448 (798)	total: 10.5s	remaining: 2.62s
900:	learn: 5242.3244370	test: 6184.4868020	best: 6182.8744902 (886)	total: 11.8s	remaining: 1.3s
999:	learn: 5201

CatBoostRegressor(depth=6, iterations=1000, learning_rate=0.05, loss_function='MAE', random_seed=42, verbose=100)

In [13]:
pred_original_mae = model_original_mae.predict(X_val)
mae_original_mae = mean_absolute_error(
    y_val,
    pred_original_mae
)
print("Original Train - MAE Loss - Validation MAE:", mae_original_mae)

Original Train - MAE Loss - Validation MAE: 6172.283924763128


In [14]:
results.loc[len(results)] = {
    "Experiment": "Original Train - MAE Loss",
    "Train Rows": len(train_original),
    "Validation MAE": mae_original_mae
}
results

,Experiment,Train Rows,Validation MAE
0,Original Train,5165,14288.439066
1,Clean Train,5109,14566.000517
2,Original Train - MAE Loss,5165,6172.283925


In [15]:
model_clean_mae = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    verbose=100,
    random_seed=42
)
model_clean_mae.fit(
    X_train_clean,
    y_train_clean,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100
)

0:	learn: 16072.3610193	test: 15002.2939225	best: 15002.2939225 (0)	total: 8.23ms	remaining: 8.22s
100:	learn: 5799.5796552	test: 6961.0911866	best: 6960.8442608 (98)	total: 577ms	remaining: 5.14s
200:	learn: 5350.9124024	test: 6600.9408168	best: 6600.9408168 (200)	total: 1.17s	remaining: 4.65s
300:	learn: 5120.8981526	test: 6354.6846571	best: 6354.6846571 (300)	total: 1.8s	remaining: 4.18s
400:	learn: 4975.0090607	test: 6292.0609963	best: 6290.7231171 (392)	total: 2.62s	remaining: 3.92s
500:	learn: 4872.8113300	test: 6277.7055014	best: 6272.2374102 (486)	total: 3.47s	remaining: 3.46s
600:	learn: 4799.7969082	test: 6261.3294642	best: 6260.1421721 (580)	total: 4.34s	remaining: 2.88s
700:	learn: 4740.3014065	test: 6253.1884330	best: 6253.1884330 (700)	total: 5.03s	remaining: 2.14s
800:	learn: 4678.3256656	test: 6253.2244242	best: 6249.9238278 (780)	total: 5.57s	remaining: 1.38s
900:	learn: 4636.0224079	test: 6245.4991794	best: 6245.4991794 (900)	total: 6.09s	remaining: 669ms
999:	learn: 

CatBoostRegressor(depth=6, iterations=1000, learning_rate=0.05, loss_function='MAE', random_seed=42, verbose=100)

In [16]:
pred_clean_mae = model_clean_mae.predict(X_val)
mae_clean_mae = mean_absolute_error(
    y_val,
    pred_clean_mae
)
print("Clean Train - MAE Loss - Validation MAE:", mae_clean_mae)

Clean Train - MAE Loss - Validation MAE: 6232.817376568046


In [17]:
results.loc[len(results)] = {
    "Experiment": "Clean Train - MAE Loss",
    "Train Rows": len(train_clean),
    "Validation MAE": mae_clean_mae
}

results

,Experiment,Train Rows,Validation MAE
0,Original Train,5165,14288.439066
1,Clean Train,5109,14566.000517
2,Original Train - MAE Loss,5165,6172.283925
3,Clean Train - MAE Loss,5109,6232.817377


**Improvement hyber**

In [34]:
deltas = [1000, 3000, 5000, 10000]
huber_results = []
for delta in deltas:
    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function=f"Huber:delta={delta}",
        random_seed=42,
        verbose=False
    )
    model.fit(
        X_train_original,
        y_train_original,
        cat_features=categorical_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )

    predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)

    huber_results.append({
        "delta": delta,
        "validation_MAE": mae,
        "best_iteration": model.best_iteration_
    })

huber_results_df = pd.DataFrame(huber_results)

display(
    huber_results_df.sort_values("validation_MAE")
)

,delta,validation_MAE,best_iteration
3,10000,6164.897976,999
1,3000,6286.544489,969
2,5000,6375.292682,999
0,1000,6502.074111,997


In [35]:
deltas = [7000, 8000, 9000, 10000, 12000, 15000, 20000]

huber_results_extended = []

for delta in deltas:

    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function=f"Huber:delta={delta}",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_original,
        y_train_original,
        cat_features=categorical_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=150,
        verbose=False
    )

    predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)

    huber_results_extended.append({
        "delta": delta,
        "validation_MAE": mae,
        "best_iteration": model.best_iteration_
    })

huber_extended_df = pd.DataFrame(huber_results_extended)

display(
    huber_extended_df.sort_values("validation_MAE")
)

,delta,validation_MAE,best_iteration
3,10000,6131.655183,1749
2,9000,6163.043462,1544
1,8000,6167.481341,1832
4,12000,6229.266937,1809
6,20000,6274.455183,1303
0,7000,6280.617165,1977
5,15000,6313.494260,1040


In [36]:
deltas = [9000, 9500, 9750, 10000, 10250, 10500, 11000]

huber_fine_results = []

for delta in deltas:

    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.05,
        depth=6,
        loss_function=f"Huber:delta={delta}",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_original,
        y_train_original,
        cat_features=categorical_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=200,
        verbose=False
    )

    predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)

    huber_fine_results.append({
        "delta": delta,
        "validation_MAE": mae,
        "best_iteration": model.best_iteration_
    })

huber_fine_df = pd.DataFrame(huber_fine_results)

display(
    huber_fine_df.sort_values("validation_MAE")
)

,delta,validation_MAE,best_iteration
1,9500,6123.441289,2074
3,10000,6131.655183,1749
6,11000,6150.752763,1254
2,9750,6151.632394,2003
0,9000,6163.043462,1544
5,10500,6178.370828,2581
4,10250,6183.663482,1043


In [37]:
depth_values = [4, 5, 6, 7, 8]

depth_results = []

for depth in depth_values:

    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.05,
        depth=depth,
        loss_function="Huber:delta=9500",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_original,
        y_train_original,
        cat_features=categorical_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=200,
        verbose=False
    )

    predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)

    depth_results.append({
        "depth": depth,
        "validation_MAE": mae,
        "best_iteration": model.best_iteration_
    })

depth_results_df = pd.DataFrame(depth_results)

display(
    depth_results_df.sort_values("validation_MAE")
)

,depth,validation_MAE,best_iteration
3,7,6108.716710,1837
2,6,6123.441289,2074
1,5,6147.421306,2289
4,8,6158.515179,1181
0,4,6235.807426,1688


In [38]:
learning_rates = [0.01, 0.02, 0.03, 0.05, 0.07, 0.1]

lr_results = []

for lr in learning_rates:

    model = CatBoostRegressor(
        iterations=4000,
        learning_rate=lr,
        depth=7,
        loss_function="Huber:delta=9500",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_original,
        y_train_original,
        cat_features=categorical_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=250,
        verbose=False
    )

    predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)

    lr_results.append({
        "learning_rate": lr,
        "validation_MAE": mae,
        "best_iteration": model.best_iteration_
    })

lr_results_df = pd.DataFrame(lr_results)

display(
    lr_results_df.sort_values("validation_MAE")
)

,learning_rate,validation_MAE,best_iteration
3,0.05,6108.716710,1837
2,0.03,6128.633529,2824
4,0.07,6188.180323,1545
0,0.01,6208.115782,3998
5,0.10,6232.825366,679
1,0.02,6257.828688,2510


In [39]:
import pandas as pd
from pathlib import Path
DATA_DIR = Path("/content/project/DEPI-GRADUATION-PROJECT-task-2/data/processed/price_review_task4")
best_result = pd.DataFrame([{
    "model": "CatBoostRegressor",
    "loss_function": "Huber",
    "delta": 9500,
    "depth": 7,
    "learning_rate": 0.05,
    "iterations": 4000,
    "best_iteration": 1837,
    "validation_MAE": 6108.716710
}])

best_path = DATA_DIR / "task_4_4_best_model.csv"

best_result.to_csv(
    best_path,
    index=False,
    encoding="utf-8-sig"
)

display(best_result)

print("Saved to:", best_path)

,model,loss_function,delta,depth,learning_rate,iterations,best_iteration,validation_MAE
0,CatBoostRegressor,Huber,9500,7,0.05,4000,1837,6108.71671


Saved to: /content/project/DEPI-GRADUATION-PROJECT-task-2/data/processed/price_review_task4/task_4_4_best_model.csv


In [40]:
experiment_tables = {
    "huber_initial": huber_results_df,
    "huber_extended": huber_extended_df,
    "huber_fine": huber_fine_df,
    "depth_comparison": depth_results_df,
    "learning_rate_comparison": lr_results_df
}

for name, df in experiment_tables.items():

    file_path = DATA_DIR / f"task_4_4_{name}.csv"

    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", file_path.name)

Saved: task_4_4_huber_initial.csv
Saved: task_4_4_huber_extended.csv
Saved: task_4_4_huber_fine.csv
Saved: task_4_4_depth_comparison.csv
Saved: task_4_4_learning_rate_comparison.csv


In [41]:
from google.colab import files
best_result = pd.DataFrame([{
    "model": "CatBoostRegressor",
    "loss_function": "Huber",
    "delta": 9500,
    "depth": 7,
    "learning_rate": 0.05,
    "iterations": 4000,
    "best_iteration": 1837,
    "validation_MAE": 6108.716710
}])

best_result.to_csv(
    "task_4_4_best_model.csv",
    index=False,
    encoding="utf-8-sig"
)

files.download("task_4_4_best_model.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>